In [1]:
%run calendario.ipynb

(909, 9)
       articolo     brand        fornitore modalita_acquisto categoria    colore stagione  prezzo_listino  costo_unitario
0  AGO-CAM-0001  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia     verde     PE23           109.0           40.47
1  AGO-CAM-0002  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia     beige     PE23           119.0           47.71
2  AGO-CAM-0003  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia  fantasia     PE23           129.0           48.24
3  AGO-CAM-0004  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia     beige     PE23            79.0           30.54
4  AGO-CAM-0005  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia    bianco     PE23           139.0           52.70
5  AGO-CAM-0006  Ago&Filo  Ago&Filo S.r.l.             fermo   camicia     beige     PE23           129.0           49.90
6  AGO-CRA-0007  Ago&Filo  Ago&Filo S.r.l.             fermo  cravatta     rosso     PE23            79.0           31.77
7  AGO-CRA-0008

In [2]:
METEO = {
    ("AI24", "capospalla"): 0.70, ("AI24", "maglia"): 0.85,
    ("PE24", "costume"): 1.35, ("PE24", "t-shirt"): 1.15,
    ("AI25", "capospalla"): 1.20,
}
SPINTA_SALDO = 2.0
ESCLUSI_DAI_SALDI = {"profumo"}
QUOTA_CLIENTI_ABITUALI = 0.05
SCONTO_CLIENTE_ABITUALE = 0.10
QUOTA_CARTA = 0.70
PROB_CAPO_IN_PIU = 0.30 

In [3]:
def fine_stagioni(cal):
    """Dopo l'ultimo giorno di saldi la merce della stagione va all'outlet o torna al fornitore."""
    return cal[cal["stagione_in_saldo"].notna()].groupby("stagione_in_saldo")["data"].max()

In [4]:
def crea_magazzino(ordini):
    mag = (ordini.groupby(["articolo", "taglia"], as_index=False)
                 .agg(qta=("qta_ricevuta", "sum"), data_arrivo=("data_arrivo", "min")))
    info = cat.set_index("articolo")[["brand", "categoria", "colore", "stagione",
                                      "prezzo_listino"]]
    mag = mag.join(info, on="articolo")
    mag["appeal"] = mag["brand"].map(BRAND.set_index("brand")["appeal"])
    pop = {}
    for categoria, scala in SCALA_PER_CATEGORIA.items():
        taglie, pesi = SCALE[scala]
        pop.update({(categoria, t): p for t, p in zip(taglie, pesi)})
    mag["pop_taglia"] = [pop[(c, t)] for c, t in zip(mag["categoria"], mag["taglia"])]
    mag["meteo"] = [METEO.get((s, c), 1.0) for s, c in zip(mag["stagione"], mag["categoria"])]
    mag["fine"] = mag["stagione"].map(fine_stagioni(cal))
    mag["data_arrivo"] = pd.to_datetime(mag["data_arrivo"])
    return mag

In [5]:
def ora_vendita(giorno):
    """Mattina 10-13 (chiusa il lunedì), pomeriggio 16:30-20; il pomeriggio vende di più."""
    mattina = giorno.day_of_week != 0 and rng.random() < 0.4
    inizio, durata = (10 * 60, 180) if mattina else (16 * 60 + 30, 210)
    minuti = inizio + int(rng.integers(durata))
    return giorno + pd.Timedelta(minutes=minuti)

In [6]:
OBIETTIVO_ST = 0.75
FORZA = 0.5
LIMITI = (0.70, 1.20)

def data_decisione(stagione):
    """Quando il titolare ordina: PE a metà settembre dell'anno prima, AI a metà febbraio."""
    anno = 2000 + int(stagione[2:])
    return pd.Timestamp(f"{anno - 1}-09-15") if stagione.startswith("PE") else pd.Timestamp(f"{anno}-02-15")

def stagione_precedente(stagione):
    """La stessa stagione dell'anno prima: AI25 -> AI24."""
    return f"{stagione[:2]}{int(stagione[2:]) - 1}"
 
def calcola_fattori(mag, ricevuti, qta, stagione):
    """Sell-through di una stagione per brand e categoria, con quello che si sa oggi."""
    sel = (mag["stagione"] == stagione).to_numpy()
    m = mag[sel].assign(ricevuti=ricevuti[sel], rimasti=qta[sel])
    g = m.groupby(["brand", "categoria"])[["ricevuti", "rimasti"]].sum()
    st = 1 - g["rimasti"] / g["ricevuti"]
    fattori = (1 + FORZA * (st - OBIETTIVO_ST)).clip(*LIMITI)
    return fattori.to_dict(), st

In [7]:
P_RESO_TAGLIA = 0.012
P_RESO_RIPENSAMENTO = 0.006
P_RESO_REGALO = 0.004
QUOTA_DIFETTI_RECLAMATI = 0.85
QUOTA_CAMBIO = 0.60
SENZA_TAGLIA = {"cravatta", "profumo"}
NON_CAMBIABILI = {"costume", "profumo"}

VALIDITA_BUONO = 180 
QUOTA_BUONI_USATI = 0.85
P_BUONO_REGALO = {12: 0.50, "altri": 0.04}
TAGLI_BUONO_REGALO = [50, 100, 150]

P_RIORDINO = 0.60
MAX_RIORDINI = 2
GIORNI_MINIMI_AI_SALDI = 21
 
vestibilita = BRAND.set_index("brand")["vestibilita"].to_dict()
difetti = BRAND.set_index("brand")["tasso_difetti"].to_dict()
fill = BRAND.set_index("brand")["fill_rate"].to_dict()
anagrafica = cat.set_index("articolo")
 
 
def estrai_reso(vendita, giorno):
    """Decide se una vendita tornerà indietro, perché e quando. None = nessun reso."""
    cat_, brand = vendita["categoria"], vendita["brand"]
    p_difetto = difetti[brand]
    p_taglia = 0 if (cat_ in SENZA_TAGLIA or cat_ in NON_CAMBIABILI) else \
        P_RESO_TAGLIA * (2 if vestibilita[brand] in ("veste piccolo", "veste grande") else 1)
    p_ripens = 0 if cat_ in NON_CAMBIABILI else P_RESO_RIPENSAMENTO
    p_regalo = 0 if cat_ in NON_CAMBIABILI else P_RESO_REGALO * (3 if giorno.month == 12 else 1)
    r = rng.random()
    for motivo, p in [("difetto", p_difetto), ("taglia", p_taglia),
                      ("ripensamento", p_ripens), ("regalo", p_regalo)]:
        if r < p:
            break
        r -= p
    else:
        return None
    fascia = rng.random()
    giorni = (rng.integers(1, 8) if fascia < 0.70 else
              rng.integers(8, 15) if fascia < 0.95 else rng.integers(15, 41))
    return motivo, giorno + pd.Timedelta(days=int(giorni))

In [8]:
def simula(cal):
    primo_anno = [s for s in STAGIONI if data_decisione(s) < cal["data"].min()]
    ordini_tutti = pd.concat([crea_ordini(cat[cat["stagione"] == s]) for s in primo_anno],
                             ignore_index=True)
    mag = crea_magazzino(ordini_tutti)
    qta = mag["qta"].to_numpy().copy()
    ricevuti = qta.copy()                         
    riordini = np.zeros(len(qta), int)
    da_decidere = [s for s in STAGIONI if s not in primo_anno]
    inizio_saldi = cal[cal["settimana_saldo"] == 1].groupby("stagione_in_saldo")["data"].min()
    ultimo_giorno = cal["data"].max()
 
    eventi = {}                                 
    def programma(giorno, evento):
        if giorno <= ultimo_giorno:
            eventi.setdefault(giorno, []).append(evento)
 
    diario, riassortimenti, righe = [], [], []
    contatore = {"scontrino": 0, "uid": 0, "buono": 0}
    def nuovo(chiave):
        contatore[chiave] += 1
        return contatore[chiave]
 
    def emetti_buono(valore, giorno, prefisso):
        codice = f"{prefisso}{nuovo('buono'):05d}"
        if rng.random() < QUOTA_BUONI_USATI:
            programma(giorno + pd.Timedelta(days=int(rng.integers(1, VALIDITA_BUONO + 1))),
                      ("riscatto", codice))
        return codice
 
    def riga_vendita(i, ora, scontrino, prezzo, sconto, motivo, metodo, buono=None):
        r = mag.iloc[i]
        riga = {"_uid": nuovo("uid"), "_i": i, "data_ora": ora, "scontrino": scontrino,
                "articolo": r["articolo"], "brand": r["brand"], "categoria": r["categoria"],
                "taglia": r["taglia"], "colore": r["colore"], "stagione": r["stagione"],
                "prezzo_listino": r["prezzo_listino"], "prezzo_venduto": prezzo,
                "sconto_pct": round(sconto * 100), "motivo_sconto": motivo,
                "tipo_movimento": "vendita", "metodo_pagamento": metodo, "id_buono": buono}
        righe.append(riga)
        return riga
 
    def forse_riordina(i, giorno):
        s = mag.at[i, "stagione"]
        if (qta[i] > 0 or riordini[i] >= MAX_RIORDINI or rng.random() >= P_RIORDINO
                or giorno > inizio_saldi[s] - pd.Timedelta(days=GIORNI_MINIMI_AI_SALDI)):
            return
        r, a = mag.iloc[i], anagrafica.loc[mag.at[i, "articolo"]]
        ordinati = max(1, int(round(mag.at[i, "qta"] * rng.uniform(0.3, 0.6))))
        arrivati = ordinati
        if fill[r["brand"]] < 1 and rng.random() < 0.4:
            arrivati = int(round(ordinati * rng.uniform(0.3, 0.7)))
        arrivo = giorno + pd.Timedelta(days=int(rng.integers(10, 21)))
        riordini[i] += 1
        programma(arrivo, ("arrivo", i, arrivati))
        riassortimenti.append({
            "articolo": r["articolo"], "taglia": r["taglia"], "brand": r["brand"],
            "fornitore": a["fornitore"], "modalita_acquisto": a["modalita_acquisto"],
            "categoria": r["categoria"], "colore": r["colore"], "stagione": s,
            "tipo_ordine": "riassortimento", "data_ordine": giorno.date(), "data_arrivo": arrivo.date(),
            "qta_ordinata": ordinati, "qta_ricevuta": arrivati,
            "costo_unitario": a["costo_unitario"], "prezzo_listino": r["prezzo_listino"]})
 
    for g in cal.itertuples(index=False):
        giorno = g.data
 
        for s in [s for s in da_decidere if data_decisione(s) <= giorno]:
            fattori, st = calcola_fattori(mag, ricevuti, qta, stagione_precedente(s))
            nuovi = crea_ordini(cat[cat["stagione"] == s], fattori)
            ordini_tutti = pd.concat([ordini_tutti, nuovi], ignore_index=True)
            nuovo_mag = crea_magazzino(nuovi)
            mag = pd.concat([mag, nuovo_mag], ignore_index=True)
            qta = np.concatenate([qta, nuovo_mag["qta"].to_numpy()])
            ricevuti = np.concatenate([ricevuti, nuovo_mag["qta"].to_numpy()])
            riordini = np.concatenate([riordini, np.zeros(len(nuovo_mag), int)])
            diario += [{"stagione": s, "brand": b, "categoria": c,
                        "sell_through_anno_prima": round(st[(b, c)], 2),
                        "fattore": round(f, 2)} for (b, c), f in fattori.items()]
            da_decidere.remove(s)
 
        in_negozio = ((mag["data_arrivo"].to_numpy() <= np.datetime64(giorno))
                      & (np.datetime64(giorno) <= mag["fine"].to_numpy()))
        buoni_oggi = []
        for evento in eventi.pop(giorno, []):
            if evento[0] == "arrivo":
                _, i, q = evento
                qta[i] += q
                ricevuti[i] += q
            elif evento[0] == "riscatto":
                buoni_oggi.append(evento[1])
            elif evento[0] == "reso":
                _, v, motivo = evento
                i = v["_i"]
                scontrino = f"S{giorno.year}-{nuovo('scontrino'):06d}"
                ora = ora_vendita(giorno)
                reso = {**v, "_uid": nuovo("uid"), "_rif": v["_uid"], "data_ora": ora,
                        "scontrino": scontrino, "tipo_movimento": "reso", "metodo_pagamento": None,
                        "id_buono": None, "motivo_reso": motivo,
                        "reso_fornitore": ("sì" if rng.random() < QUOTA_DIFETTI_RECLAMATI else "no")
                                          if motivo == "difetto" else None}
                if motivo != "difetto" and in_negozio[i]:
                    qta[i] += 1 
                candidati = [j for j in np.flatnonzero((mag["articolo"].to_numpy() == v["articolo"])
                                                        & (qta > 0) & in_negozio)
                             if (j == i) == (motivo == "difetto")]
                if motivo in ("taglia", "difetto") and candidati and rng.random() < QUOTA_CAMBIO:
                    reso["esito_reso"] = "cambio"
                    righe.append(reso)
                    j = int(rng.choice(candidati))
                    qta[j] -= 1
                    riga_vendita(j, ora, scontrino, v["prezzo_venduto"], v["sconto_pct"] / 100,
                                 v["motivo_sconto"], "cambio")
                    forse_riordina(j, giorno)
                else:
                    reso["esito_reso"] = "buono"
                    reso["id_buono"] = emetti_buono(v["prezzo_venduto"], giorno, "B")
                    righe.append(reso)
 
        if rng.random() < P_BUONO_REGALO.get(giorno.month, P_BUONO_REGALO["altri"]):
            valore = float(rng.choice(TAGLI_BUONO_REGALO))
            righe.append({"_uid": nuovo("uid"), "data_ora": ora_vendita(giorno),
                          "scontrino": f"S{giorno.year}-{nuovo('scontrino'):06d}",
                          "prezzo_listino": valore, "prezzo_venduto": valore, "sconto_pct": 0,
                          "motivo_sconto": "nessuno", "tipo_movimento": "buono_regalo",
                          "metodo_pagamento": "carta" if rng.random() < QUOTA_CARTA else "contanti",
                          "id_buono": emetti_buono(valore, giorno, "G")})
 
        peso_base = (mag["appeal"] * mag["pop_taglia"] * mag["meteo"]).to_numpy()
        in_saldo = (((mag["stagione"].to_numpy() == g.stagione_in_saldo)
                     & ~mag["categoria"].isin(ESCLUSI_DAI_SALDI).to_numpy()) if g.sconto_saldo > 0
                    else np.zeros(len(qta), bool))
        peso = peso_base * in_negozio * (1 + SPINTA_SALDO * g.sconto_saldo * in_saldo)
        da_vendere = int(g.capi)
        while da_vendere > 0:
            scontrino = f"S{giorno.year}-{nuovo('scontrino'):06d}"
            ora = ora_vendita(giorno)
            pezzi = min(da_vendere, int(rng.geometric(1 - PROB_CAPO_IN_PIU)))
            buono = buoni_oggi.pop() if buoni_oggi else None
            metodo = "buono" if buono else ("carta" if rng.random() < QUOTA_CARTA else "contanti")
            for k in range(pezzi):
                p = peso * (qta > 0)
                if p.sum() == 0:
                    break
                i = int(rng.choice(len(qta), p=p / p.sum()))
                qta[i] -= 1
                listino = mag.at[i, "prezzo_listino"]
                if in_saldo[i]:
                    sconto, motivo = g.sconto_saldo, "saldo"
                elif rng.random() < QUOTA_CLIENTI_ABITUALI:
                    sconto, motivo = SCONTO_CLIENTE_ABITUALE, "cliente_abituale"
                else:
                    sconto, motivo = 0.0, "nessuno"
                v = riga_vendita(i, ora, scontrino, round(listino * (1 - sconto), 2), sconto, motivo,
                                 metodo, buono if k == 0 else None)
                esito = estrai_reso(v, giorno)
                if esito:
                    programma(esito[1], ("reso", v, esito[0]))
                forse_riordina(i, giorno)
            da_vendere -= max(pezzi, 1)
        for b in buoni_oggi:
            programma(giorno + pd.Timedelta(days=1), ("riscatto", b))
 
    mag = mag.assign(qta=ricevuti, qta_finale=qta, riordini=riordini)
    ordini_tutti = pd.concat([ordini_tutti, pd.DataFrame(riassortimenti)], ignore_index=True)
    vendite = pd.DataFrame(righe).sort_values("data_ora", kind="stable", ignore_index=True)
    primo = vendite.groupby("scontrino")["data_ora"].min().sort_values(kind="stable")
    progressivo = primo.groupby(primo.dt.year).cumcount() + 1
    nuovo_num = {s: f"S{d.year}-{n:06d}" for s, d, n in zip(primo.index, primo, progressivo)}
    vendite["scontrino"] = vendite["scontrino"].map(nuovo_num)
    vendite = vendite.sort_values(["data_ora", "scontrino"], kind="stable", ignore_index=True)
    vendite.insert(0, "id_riga", range(1, len(vendite) + 1))
    da_uid = dict(zip(vendite["_uid"], vendite["id_riga"]))
    vendite["rif_vendita"] = vendite["_rif"].map(da_uid).astype("Int64")
    colonne = ["id_riga", "data_ora", "scontrino", "articolo", "brand", "categoria", "taglia",
               "colore", "stagione", "prezzo_listino", "prezzo_venduto", "sconto_pct",
               "motivo_sconto", "tipo_movimento", "rif_vendita", "esito_reso", "motivo_reso",
               "metodo_pagamento", "id_buono", "reso_fornitore"]
    return vendite[colonne], mag, ordini_tutti, pd.DataFrame(diario)
     

In [9]:
vendite, mag, ordini, diario = simula(cal)
print(vendite.shape)
print(vendite["tipo_movimento"].value_counts())
 
print("\nMagazzino mai negativo:", (mag["qta_finale"] >= 0).all())
vend = vendite[vendite["tipo_movimento"] == "vendita"]
resi = vendite[vendite["tipo_movimento"] == "reso"]
originali = vendite.set_index("id_riga").loc[resi["rif_vendita"]]
print("Ogni reso punta a una vendita dello stesso articolo:",
      (originali["articolo"].to_numpy() == resi["articolo"].to_numpy()).all()
      and (originali["tipo_movimento"] == "vendita").all())
 
print("\nTasso di resi per brand (Dipartimento 3 il più alto: difetti):")
print((resi.groupby("brand").size() / vend.groupby("brand").size()).sort_values(ascending=False).round(3))
print("\nResi per taglia sulle vendite (Lardoni, Cruciverba, Dipartimento 3 in alto):")
print((resi[resi["motivo_reso"] == "taglia"].groupby("brand").size()
       / vend.groupby("brand").size()).sort_values(ascending=False).round(3))
 
giorni_reso = (resi["data_ora"].dt.normalize().to_numpy()
               - originali["data_ora"].dt.normalize().to_numpy()) / np.timedelta64(1, "D")
print("\nResi oltre 14 giorni:", int((giorni_reso > 14).sum()), "su", len(resi))
print("Difetti non rimandati al fornitore:", int((resi["reso_fornitore"] == "no").sum()))
print("Esito dei resi:", resi["esito_reso"].value_counts().to_dict())
 
print("\nRiassortimenti per brand (Lieve in alto):")
ria = ordini[ordini["tipo_ordine"] == "riassortimento"]
print(ria.groupby("brand").size().sort_values(ascending=False))
st_brand = vend.groupby("brand").size() / mag.groupby("brand")["qta"].sum()
print("\nSell-through per brand:")
print(st_brand.sort_values().round(2))

emessi = vendite[vendite["tipo_movimento"].isin(["reso", "buono_regalo"]) & vendite["id_buono"].notna()]
spesi = vendite[(vendite["metodo_pagamento"] == "buono")]
lieve = vend[vend["brand"] == "Lieve"]
print("\nProfumi venduti in saldo (deve essere 0):", int((lieve["motivo_sconto"] == "saldo").sum()))
print("Profumi con sconto al cliente abituale:", int((lieve["motivo_sconto"] == "cliente_abituale").sum()))
print("\nBuoni emessi:", len(emessi), "— spesi:", spesi["id_buono"].nunique())
print("\nIl diario del titolare — FreePlace, capospalla:")
print(diario[(diario["brand"] == "FreePlace Outerwear") & (diario["categoria"] == "capospalla")])

(16433, 20)
tipo_movimento
vendita         15838
reso              502
buono_regalo       93
Name: count, dtype: int64

Magazzino mai negativo: True
Ogni reso punta a una vendita dello stesso articolo: True

Tasso di resi per brand (Dipartimento 3 il più alto: difetti):
brand
Lardoni                0.047
Dipartimento 3         0.045
Hole                   0.041
Cruciverba             0.040
Borris Calzature       0.031
Grillo                 0.022
Ago&Filo               0.020
Lido Club              0.019
FreePlace Outerwear    0.016
Lieve                  0.006
dtype: float64

Resi per taglia sulle vendite (Lardoni, Cruciverba, Dipartimento 3 in alto):
brand
Lardoni                0.030
Cruciverba             0.029
Dipartimento 3         0.022
Hole                   0.017
Borris Calzature       0.012
Grillo                 0.011
Lido Club              0.008
Ago&Filo               0.006
FreePlace Outerwear    0.005
Lieve                    NaN
dtype: float64

Resi oltre 14 giorni: 21 su 